In [1]:
import pandas as pd
import sys
sys.path.append('../src')

from preprocessing import combine_datasets, parse_protein_go_data, save_tsv, trim_go_terms

In [2]:
def load_preds_tsv(path, conf_col="confidence"):
    """Load one model's predictions. Deduplicates on (protein, term) via max."""
    df = pd.read_csv(
        path, sep='\t', header=None,
        names=['protein_id', 'GO_term', "confidence"],
    )
    return df

In [3]:
structure_df = load_preds_tsv('/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/structure_team/clean/CAFA_trim_thr-50/train_predictions.tsv')
protgoat_df = load_preds_tsv('/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/ProtGOAT/protgoat_train_predictions.tsv')

In [4]:
display(structure_df.head())

,protein_id,GO_term,confidence
0,A0A0C9S461,GO:0008150,0.827095
1,A0A0C9S461,GO:0003674,0.811765
2,A0A0C9S461,GO:0005488,0.811593
3,A0A0C9S461,GO:0005515,0.811593
4,A0A0C9S461,GO:0050896,0.704853


In [5]:
# get the set of unique protein IDs from both dataframes
structure_proteins = set(structure_df['protein_id'])
protgoat_proteins = set(protgoat_df['protein_id'])

# check if proteins match and get ones that don't match
matching_proteins = structure_proteins.intersection(protgoat_proteins)
missing_in_structure = protgoat_proteins - structure_proteins
print(f"Number of proteins in structure_df: {len(structure_proteins)}")
print(f"Number of proteins in protgoat_df: {len(protgoat_proteins)}")
print(f"Number of matching proteins: {len(matching_proteins)}")
print(f"Proteins in protgoat_df but missing in structure_df: {missing_in_structure}")
# If there are missing proteins, you can decide how to handle them (e.g., ignore, fill with NaN, etc.)

Number of proteins in structure_df: 126932
Number of proteins in protgoat_df: 142245
Number of matching proteins: 126932
Proteins in protgoat_df but missing in structure_df: {'Q502F5', 'P0C739', 'B2LUM7', 'E9QEG1', 'E9QC94', 'A0A2R8Q5U4', 'X1WG08', 'Q7ZUU3', 'A0A0B4K7Y9', 'E9QGF0', 'O82629', 'A0A0R4IFV9', 'Q90ZR5', 'R4GEV4', 'I3ITG5', 'F1QAS9', 'E7ER32', 'Q8JIY3', 'Q6IQB9', 'A0A2R8QLY8', 'Q8JHH9', 'Q2YDS8', 'H0WF03', 'A0A024R794', 'F6NJE9', 'Q6AYN8', 'X1WBM9', 'P86562', 'A0A0A0MT47', 'Q1MSX5', 'Q6DC06', 'D6QWX7', 'Q32LV7', 'A0A3T1CVI5', 'E7FGM4', 'E9QCV2', 'P90495', 'A0A2R8QT61', 'Q6DRJ5', 'Q99719', 'P21607', 'Q9VJJ9', 'H7C0W6', 'A0A1D5NSI9', 'Q5SPR2', 'E9QBS8', 'F1R632', 'F6P1L8', 'Q2F6M4', 'H7BXY5', 'Q9Y1V0', 'Q4VV68', 'E1JGZ8', 'A0A2R8QA90', 'O95235', 'D4A4T0', 'O08550', 'T1TF52', 'Q9PTR4', 'A0A1W2PQD4', 'Q6B334', 'M9MRD6', 'Q2VA53', 'Q5XIY3', 'Q6XK18', 'P84273', 'E1BR22', 'F1RBX3', 'B2XY78', 'Q00174', 'M9ND93', 'A2BEN1', 'Q0E9K6', 'A0A0R4IX63', 'A0A1D5PVE2', 'P86309', 'M9PHF1', 'Q9

In [13]:
import pandas as pd
import re
import os

COLS = ['protein_id', 'go_term', 'confidence']

def load_tsv(tsv_path):
    if tsv_path.endswith('.gz'):
        df = pd.read_csv(tsv_path, sep='\t', header=None, compression='gzip')
    else:
        df = pd.read_csv(tsv_path, sep='\t', header=None)
    df = df.iloc[:, :3]  # take only first 3 columns
    df.columns = COLS
    return df

def combine_datasets(datasets):
    return pd.concat(datasets, ignore_index=True)[COLS]

def save_tsv(df, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df[COLS].to_csv(output_path, sep='\t', index=False, header=False)

def trim_go_terms(df, threshold_csv="..."):
    frequency_df = pd.read_csv(threshold_csv)
    frequency_set = set(frequency_df.iloc[:, 0])
    return df[df['go_term'].isin(frequency_set)]

def parse_protein_go_data(df):
    """Extract protein ID, GO term, and confidence score from annotation dataframe."""
    # Assuming the data is in a single column, split it appropriately
    df['protein_id'] = df.iloc[:, 0].str.split().str[0]
    df['go_term'] = df.iloc[:, 0].str.extract(r'(GO:\d+)')[0]
    df['confidence_score'] = df.iloc[:, 0].str.extract(r'(GO:\d+\s+([\d.]+))')[1]
    
    return df[['protein_id', 'go_term', 'confidence_score']]

def parse_protein_go_data(df):
    """Extract protein ID, GO term, confidence from space-separated single column."""
    # Split the single column on whitespace
    parts = df.iloc[:, 0].str.split(r'\s+', expand=True)
    
    # Validate we got exactly 3 columns
    if parts.shape[1] != 3:
        raise ValueError(f"Expected 3 columns after split, got {parts.shape[1]}")
    
    parts.columns = COLS
    
    # Validate GO term format
    invalid_go = ~parts['go_term'].str.match(r'^GO:\d+$')
    if invalid_go.any():
        print(f"Warning: {invalid_go.sum()} invalid GO terms found and removed")
        parts = parts[~invalid_go]
    
    # Convert confidence to float
    parts['confidence'] = pd.to_numeric(parts['confidence'], errors='coerce')
    
    # Drop any rows with NaN after conversion
    if parts['confidence'].isna().any():
        print(f"Warning: {parts['confidence'].isna().sum()} rows with invalid confidence, removing")
        parts = parts[parts['confidence'].notna()]
    
    return parts

In [16]:
COLS = ['protein_id', 'go_term', 'confidence']
def load_tsv(tsv_path):
    raw_df = pd.read_csv(tsv_path, sep='\t', header=None)

    # Parse it
    parsed_df = parse_protein_go_data(raw_df)

    print(parsed_df.head())
    print(f"Loaded {len(parsed_df)} (protein, GO term, confidence) triplets")
    return parsed_df

train_bp = load_tsv("/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/raw/thr-50_train_val/train_50/f0_train_50_trim/f0_train_50_sequences_preds_bp_trim.tsv.gz")
train_cc = load_tsv("/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/raw/thr-50_train_val/train_50/f0_train_50_trim/f0_train_50_sequences_preds_cc_trim.tsv.gz")
train_mf = load_tsv("/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/raw/thr-50_train_val/train_50/f0_train_50_trim/f0_train_50_sequences_preds_mf_trim.tsv.gz")

combined_df = pd.concat([train_bp, train_cc, train_mf], ignore_index=True)

# For duplicate (protein_id, go_term) pairs, keep the max confidence
combined_df = combined_df.groupby(['protein_id', 'go_term'], as_index=False)['confidence'].max()

print(f"BP: 1974240, CC: 1002754, MF: 648968")
print(f"Total before combining: {1974240 + 1002754 + 648968}")
print(f"Combined (deduped): {len(combined_df)} unique (protein, GO term) pairs")

display(combined_df.head())
print(f"Length of combined_df: {len(combined_df)}")

missing_in_sequence = set(combined_df['protein_id']) - protgoat_proteins

save_tsv(combined_df, "/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/clean/CAFA_trim_thr-50/train_submission.tsv")

  protein_id     go_term  confidence
0     P83308  GO:0009987       0.573
1     P83308  GO:0050896       0.648
2     P0DKJ0  GO:0009987       0.650
3     P0DKJ0  GO:0050896       0.575
4     P86012  GO:0009987       0.634
Loaded 1974240 (protein, GO term, confidence) triplets
  protein_id     go_term  confidence
0     P83308  GO:0110165       0.980
1     P83308  GO:0005576       0.973
2     P0DKJ0  GO:0110165       0.980
3     P0DKJ0  GO:0005576       0.962
4     P41495  GO:0110165       0.980
Loaded 1002754 (protein, GO term, confidence) triplets
  protein_id     go_term  confidence
0     P83308  GO:0005488       0.702
1     P83308  GO:0098772       0.538
2     P83308  GO:0003824       0.629
3     P0DKJ0  GO:0005488       0.569
4     P0DKJ0  GO:0003824       0.697
Loaded 648968 (protein, GO term, confidence) triplets
BP: 1974240, CC: 1002754, MF: 648968
Total before combining: 3625962
Combined (deduped): 3625962 unique (protein, GO term) pairs


,protein_id,go_term,confidence
0,A0A009IHW8,GO:0003824,0.625
1,A0A009IHW8,GO:0003953,0.970
2,A0A009IHW8,GO:0006139,0.549
3,A0A009IHW8,GO:0006725,0.543
4,A0A009IHW8,GO:0006807,0.679


Length of combined_df: 3625962


In [ ]:
df = pd.read_csv("/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/clean/CAFA_trim_thr-50/train_submission.tsv")
print(df.head())
# print length
print(f"Length of saved TSV: {len(df)}")